In [12]:
#!/usr/bin/env python
# coding: utf-8

# ### Restructured Code for Auction Simulation and Revenue Prediction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, beta as beta_dist, entropy
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

# ========== Data Loading Functions ==========

def load_data(auction_type, filepath):
    data = pd.read_csv(filepath)
    if auction_type == 'first price':
        cleaned_data = data[(data['matched_info'] == 'yes') & 
                            (data['bid_fp'] >= 0) &
                            (data['bid_fp'] <= 30) &
                            (data['value_fp'] <= 30) &
                            (data['value_fp'] >= 0)].copy()
        cleaned_data['bid_fp'] = cleaned_data['bid_fp'] / 30  # Normalize to 0-1 range
        cleaned_data['value_fp'] = cleaned_data['value_fp'] / 30  # Normalize to 0-1 range
        return cleaned_data
    elif auction_type == 'second price':
        cleaned_data = data[(data['matched_info'] == 'yes') & 
                            (data['bid_sp'] >= 0) &
                            (data['bid_sp'] <= 45) &
                            (data['value_sp'] <= 30) &
                            (data['value_sp'] >= 0)].copy()
        cleaned_data['bid_sp'] = cleaned_data['bid_sp'] / 30  # Normalize to 0-1 range
        cleaned_data['value_sp'] = cleaned_data['value_sp'] / 30  # Normalize to 0-1 range
        return cleaned_data
    elif auction_type == 'all-pay':
        cleaned_data = data[(data['matched_info'] == 'yes') & 
                            (data['bid_ap'] >= 0) &
                            (data['bid_ap'] <= data['value_ap']) &
                            (data['value_ap'] <= 30) &
                            (data['value_ap'] >= 0)].copy()
        cleaned_data['bid_ap'] = cleaned_data['bid_ap'] / 30  # Normalize to 0-1 range
        cleaned_data['value_ap'] = cleaned_data['value_ap'] / 30  # Normalize to 0-1 range
        return cleaned_data
    else:
        raise ValueError("Unknown auction type")

# ========== Q-Learning Agent Class ==========

class QLearningAgent:
    def __init__(self, valuation_distribution, action_min=0, action_max=1, granularity=21, epsilon=0.99, alpha=0.1, total_rounds=10000):
        self.valuation_distribution = valuation_distribution
        self.actions = np.linspace(action_min, action_max, granularity)
        self.epsilon = epsilon
        self.alpha = alpha
        self.total_rounds = total_rounds
        self.granularity = granularity
        self.q_values = np.random.uniform(0,1,(granularity, len(self.actions)))
        self.valuation = self.valuation_distribution()
        self.state = self.get_state(self.valuation)
    
    def epsilon_decay(self, current_round):
        if current_round < 0.66 * self.total_rounds:
            self.epsilon = 0.99 - (0.94 * current_round / (0.66 * self.total_rounds))
        else:
            self.epsilon = 0.05
    
    def get_state(self, valuation):
        bin_centers = np.linspace(0, 1, self.granularity)
        return np.argmin(np.abs(bin_centers - valuation))
    
    def refresh_valuation(self):
        self.valuation = self.valuation_distribution()
        self.state = self.get_state(self.valuation)
    
    def choose_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.actions)
        return self.actions[np.argmax(self.q_values[self.state])]
    
    def update_q_values(self, chosen_action, reward):
        action_index = np.where(self.actions == chosen_action)[0][0]
        self.q_values[self.state, action_index] += self.alpha * (reward - self.q_values[self.state, action_index])
    
    def feedback(self, reward, action):
        self.update_q_values(action, reward)

# ========== Auction Classes and Rules ==========

def first_price_rule(bids):
    max_bid = np.max(bids)
    winner_candidates = np.where(bids == max_bid)[0]
    winner = np.random.choice(winner_candidates) if len(winner_candidates) > 1 else winner_candidates[0]
    return winner, bids[winner]

def second_price_rule(bids):
    max_bid = np.max(bids)
    winner_candidates = np.where(bids == max_bid)[0]
    winner = np.random.choice(winner_candidates) if len(winner_candidates) > 1 else winner_candidates[0]
    second_highest_bid = np.partition(bids, -2)[-2]  # Second-highest bid for payment
    return winner, second_highest_bid

def all_pay_rule(bids):
    max_bid = np.max(bids)
    winner_candidates = np.where(bids == max_bid)[0]
    winner = np.random.choice(winner_candidates) if len(winner_candidates) > 1 else winner_candidates[0]
    return winner, bids  # Return individual bids for each agent

class SingleSidedAuction:
    def __init__(self, agents, price_floor=0, price_ceiling=1, payment_rule=None, alpha=1.0, beta=1.0):
        self.agents = agents
        self.price_floor = price_floor
        self.price_ceiling = price_ceiling
        self.payment_rule = payment_rule
        self.alpha = alpha
        self.beta = beta

    def run_auction(self, current_round):
        for agent in self.agents:
            agent.epsilon_decay(current_round)
            agent.refresh_valuation()
        
        bids = [max(self.price_floor, min(agent.choose_action(), self.price_ceiling)) for agent in self.agents]
        winner_index, payment = self.payment_rule(bids)
        
        for idx, agent in enumerate(self.agents):
            if idx == winner_index:
                payoff = agent.valuation - payment
                reward = beta_dist.cdf(payoff, self.alpha, self.beta) if payoff >= 0 else payoff
                agent.feedback(reward, bids[idx])
            else:
                agent.feedback(0, bids[idx])

class AllPayAuction:
    def __init__(self, agents, price_floor=0, price_ceiling=1, payment_rule=None, alpha=1.0, beta=1.0):
        self.agents = agents
        self.price_floor = price_floor
        self.price_ceiling = price_ceiling
        self.payment_rule = payment_rule
        self.alpha = alpha
        self.beta = beta

    def run_auction(self, current_round):
        for agent in self.agents:
            agent.epsilon_decay(current_round)
            agent.refresh_valuation()
        
        bids = [max(self.price_floor, min(agent.choose_action(), self.price_ceiling)) for agent in self.agents]
        winner_index, bids = self.payment_rule(bids)
        
        for idx, agent in enumerate(self.agents):
            payoff = agent.valuation - bids[idx] if idx == winner_index else -bids[idx]  # All players pay their bid
            reward = beta_dist.cdf(payoff, self.alpha, self.beta) if payoff >= 0 else payoff
            agent.feedback(reward, bids[idx])

# ========== Simulation and Calibration Functions ==========

def run_monte_carlo_simulation(agents, auction, num_simulations=1, num_rounds=50000):
    all_bids_for_valuations = [[] for _ in range(len(agents))]
    learned_bidding_functions = [[] for _ in range(len(agents))]  # Store learned bidding functions

    for sim in range(num_simulations):
        print(sim)
        for round_num in range(num_rounds):
            auction.run_auction(round_num)
        
        for i, agent in enumerate(agents):
            avg_bids_for_valuation = [agent.actions[np.argmax(agent.q_values[state])] for state in range(agent.granularity)]
            all_bids_for_valuations[i].append(avg_bids_for_valuation)
            learned_bidding_functions[i] = avg_bids_for_valuation  # Store the learned function

    percentiles = {}
    for i in range(len(agents)):
        avg_bids_for_agent = np.array(all_bids_for_valuations[i])
        percentiles[i] = {
            "5th": np.percentile(avg_bids_for_agent, 5, axis=0),
            "median": np.median(avg_bids_for_agent, axis=0),
            "95th": np.percentile(avg_bids_for_agent, 95, axis=0),
        }
    return percentiles, learned_bidding_functions

def calculate_kl_divergence(true_bids, simulated_bids):
    true_hist, bin_edges = np.histogram(true_bids, bins=50, density=True, range=(0, 1))
    sim_hist, _ = np.histogram(simulated_bids, bins=bin_edges, density=True, range=(0, 1))
    kl_divergence = entropy(true_hist + 1e-10, sim_hist + 1e-10)
    return kl_divergence

def find_best_alpha_beta(training_data, auction_type, valuation_distribution, alpha_values, beta_values, bid_column, num_simulations=10, num_rounds=10000):
    true_bids = training_data[bid_column].values

    best_kl = float('inf')
    best_alpha = None
    best_beta = None

    for alpha in alpha_values:
        for beta in beta_values:
            # Create new agents for each alpha and beta
            agents = [QLearningAgent(valuation_distribution=valuation_distribution) for _ in range(2)]

            # Set up auction
            if auction_type == 'first price':
                auction = SingleSidedAuction(agents, payment_rule=first_price_rule, alpha=alpha, beta=beta)
            elif auction_type == 'second price':
                auction = SingleSidedAuction(agents, payment_rule=second_price_rule, alpha=alpha, beta=beta)
            elif auction_type == 'all-pay':
                auction = AllPayAuction(agents, payment_rule=all_pay_rule, alpha=alpha, beta=beta)
            else:
                raise ValueError("Unknown auction type")

            # Run simulation
            percentiles, learned_bidding_functions = run_monte_carlo_simulation(agents, auction, num_simulations=num_simulations, num_rounds=num_rounds)
            
            print(learned_bidding_functions)
            
            # Get simulated bids
            simulated_bids = percentiles[0]["median"]

            # Compute KL divergence
            kl_divergence = calculate_kl_divergence(true_bids, simulated_bids)
            print(alpha, beta, kl_divergence)

            # Check if this is the best
            if kl_divergence < best_kl:
                best_kl = kl_divergence
                best_alpha = alpha
                best_beta = beta

    return best_alpha, best_beta

def simulate_revenue_using_bidding_function(valuation_distribution, bidding_function, auction_type, num_samples=10000):
    revenues = []
    for _ in range(num_samples):
        valuations = [valuation_distribution() for _ in range(2)]
        bids = [bidding_function(v) for v in valuations]
        if auction_type == 'first price':
            revenue = max(bids)
        elif auction_type == 'second price':
            revenue = min(bids)
        elif auction_type == 'all-pay':
            revenue = sum(bids)
        else:
            raise ValueError("Unknown auction type")
        revenues.append(revenue)
    return revenues

def simulate_auction_revenue(auction_type, valuation_distribution, alpha, beta, num_simulations=1, num_rounds=100000):
    # Create agents
    agents = [QLearningAgent(valuation_distribution=valuation_distribution) for _ in range(2)]

    # Set up auction
    if auction_type == 'first price':
        auction = SingleSidedAuction(agents, payment_rule=first_price_rule, alpha=alpha, beta=beta)
    elif auction_type == 'second price':
        auction = SingleSidedAuction(agents, payment_rule=second_price_rule, alpha=alpha, beta=beta)
    elif auction_type == 'all-pay':
        auction = AllPayAuction(agents, payment_rule=all_pay_rule, alpha=alpha, beta=beta)
    else:
        raise ValueError("Unknown auction type")

    # Run simulation
    percentiles, learned_bidding_functions = run_monte_carlo_simulation(agents, auction, num_simulations=num_simulations, num_rounds=num_rounds)

    # Extract learned bidding function
    bidding_function = learned_bidding_functions[0]  # For the first agent
    
    # Interpolate the bidding function
    valuation_grid = np.linspace(0, 1, len(bidding_function))
    bidding_function_interp = lambda v: np.interp(v, valuation_grid, bidding_function)
    
    # Simulate revenues
    revenues = simulate_revenue_using_bidding_function(valuation_distribution, bidding_function_interp, auction_type)

    return revenues

def compute_actual_revenues(actual_bids, auction_type):
    num_bidders = len(actual_bids)
    revenues = []
    # Randomly pair bids
    np.random.shuffle(actual_bids)
    for i in range(0, num_bidders - 1, 2):
        bid1 = actual_bids[i]
        bid2 = actual_bids[i+1]
        bids = [bid1, bid2]
        if auction_type == 'first price':
            revenue = max(bids)
        elif auction_type == 'second price':
            revenue = min(bids)
        elif auction_type == 'all-pay':
            revenue = sum(bids)
        else:
            raise ValueError("Unknown auction type")
        revenues.append(revenue)
    return revenues

def compare_seller_revenues(simulated_revenues, actual_revenues, auction_type):
    # Plot the distributions
    plt.figure(figsize=(10, 6))
    sns.kdeplot(simulated_revenues, label='Simulated Revenues', shade=True)
    sns.kdeplot(actual_revenues, label='Actual Revenues', shade=True)
    plt.title(f'Seller Revenue Comparison for {auction_type.title()} Auction')
    plt.xlabel('Revenue')
    plt.ylabel('Density')
    plt.legend()
    plt.show()

# ========== Main Function ==========

def main(training_auction_type):
    filepath = 'clean_kaplan_data.csv'  # Update with the correct path to your data file

    # Load data for the training auction type
    training_data = load_data(training_auction_type, filepath)

    # Get the valuation column and bid column
    valuation_column = 'value_fp' if training_auction_type == 'first price' else \
                       'value_sp' if training_auction_type == 'second price' else \
                       'value_ap' if training_auction_type == 'all-pay' else None

    bid_column = 'bid_fp' if training_auction_type == 'first price' else \
                 'bid_sp' if training_auction_type == 'second price' else \
                 'bid_ap' if training_auction_type == 'all-pay' else None

    # Get valuation distribution
    valuations = training_data[valuation_column].values
    kde = gaussian_kde(valuations, bw_method='scott')
    valuation_distribution = lambda: kde.resample(1)[0][0]

    # Define alpha and beta values to search
    alpha_values = [0.1, 0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0]
    beta_values = [0.1, 0.25, 0.5, 0.75, 1.0, 2.0, 5.0, 10.0]

    # Find best alpha and beta parameters
    best_alpha, best_beta = find_best_alpha_beta(training_data, training_auction_type, valuation_distribution, alpha_values, beta_values, bid_column)
    print(f"Best parameters from training on {training_auction_type} auction: alpha={best_alpha}, beta={best_beta}")

    # For the other two auction types
    other_auction_types = ['first price', 'second price', 'all-pay']
    other_auction_types.remove(training_auction_type)

    for auction_type in other_auction_types:
        print(f"\nSimulating and comparing seller revenue for {auction_type} auction...")
        # Simulate auction revenue
        simulated_revenues = simulate_auction_revenue(auction_type, valuation_distribution, best_alpha, best_beta)

        # Load actual data
        actual_data = load_data(auction_type, filepath)
        bid_column = 'bid_fp' if auction_type == 'first price' else \
                     'bid_sp' if auction_type == 'second price' else \
                     'bid_ap' if auction_type == 'all-pay' else None

        actual_bids = actual_data[bid_column].values

        # Compute actual seller revenues
        actual_revenues = compute_actual_revenues(actual_bids, auction_type)

        # Compare the simulated revenues with actual revenues
        compare_seller_revenues(simulated_revenues, actual_revenues, auction_type)

# ========== Run the Main Function ==========

if __name__ == "__main__":
    # Specify the training auction type: 'first price', 'second price', or 'all-pay'
    training_auction_type = 'first price'  # Change this to your desired training data
    main(training_auction_type)


0
1
2
3
4
5
6
7
8
9
[[0.0, 0.0, 0.05, 0.05, 0.1, 0.2, 0.25, 0.25, 0.35000000000000003, 0.30000000000000004, 0.45, 0.45, 0.5, 0.5, 0.65, 0.6000000000000001, 0.75, 0.7000000000000001, 0.8, 0.8, 0.8500000000000001], [0.0, 0.0, 0.05, 0.05, 0.15000000000000002, 0.25, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.45, 0.45, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.6000000000000001, 0.75, 0.9, 0.9]]
0.1 0.1 16.404623914405228
0
1
2
3
4
5
6
7
8
9
[[0.0, 0.05, 0.05, 0.05, 0.15000000000000002, 0.2, 0.2, 0.25, 0.35000000000000003, 0.35000000000000003, 0.30000000000000004, 0.5, 0.45, 0.6000000000000001, 0.45, 0.65, 0.75, 0.8, 0.8500000000000001, 0.9, 0.75], [0.0, 0.0, 0.05, 0.1, 0.2, 0.2, 0.25, 0.25, 0.35000000000000003, 0.25, 0.45, 0.5, 0.55, 0.5, 0.6000000000000001, 0.7000000000000001, 0.65, 0.7000000000000001, 0.8500000000000001, 0.9, 0.9500000000000001]]
0.1 0.25 11.70040011152658
0
1
2
3
4
5
6
7
8
9
[[0.0, 0.0, 0.05, 0.15000000000000002, 0.15000000000000002, 0.2, 0.

KeyboardInterrupt: 